In [1]:
#!/usr/bin/env python3
# finetune_lora.py

"""
LoRA 파인튜닝 스크립트 (polyglot-ko-5.8b-chat)
사용법:
1) pip install torch transformers datasets accelerate peft
2) Git LFS로 polyglot-ko-5.8b-chat 클론 및 weights 다운로드
3) train_data.jsonl 준비 ({"instruction":"…","input":"…","output":"…"} 형식)
4) accelerate launch finetune_lora.py
"""

import os
import json
from datasets import load_dataset
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

# ─── 설정 ──────────────────────────────────────────────────────────────
MODEL_NAME = "/home/remote/Ai_Capstone_Project/polyglot-ko-5.8b-chat"
DATA_PATH  = "/home/remote/Ai_Capstone_Project/data_singleline.jsonl"
OUTPUT_DIR = "./lora-5.8b-chat"
BATCH_SIZE = 4
EPOCHS     = 3
LR         = 2e-4

# ─── 토크나이저 & 모델 로드 ────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# ❗️ NeoX 기반 모델은 'query_key_value'와 'dense' 모듈을 LoRA 타겟으로 사용합니다.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query_key_value", "dense"],
)
model = get_peft_model(model, lora_config)

# ─── 데이터셋 로딩 & 전처리 ───────────────────────────────────────────
# JSONL 파일: {"instruction":"…","input":"…","output":"…"}
ds = load_dataset(
    "json",
    data_files=DATA_PATH,
    split="train",
    cache_dir="./dataset_cache",  # 로컬 캐시 디렉토리 사용
    num_proc=4,  # 병렬 처리 사용
    streaming=True  # 스트리밍 모드 사용
).take(1000)  # 테스트를 위해 일부 데이터만 사용

def make_prompt(example):
    instr = example["instruction"].strip()
    inp   = example["input"].strip()
    tgt   = example["output"].strip()
    # prompt 템플릿
    prompt = f"{instr}\n\n### 사용자 질문:\n{inp}\n\n### 챗봇 답변:"
    return {"prompt": prompt, "target": tgt}

ds = ds.map(make_prompt, remove_columns=ds.column_names)

def tokenize_fn(ex):
    full = ex["prompt"] + " " + ex["target"] + tokenizer.eos_token
    tokenized = tokenizer(full, truncation=True, max_length=1024)
    input_ids = tokenized["input_ids"]
    # prompt 토큰들은 loss 계산에서 제외
    prompt_len = len(tokenizer(ex["prompt"], add_special_tokens=False)["input_ids"])
    labels = [-100] * prompt_len + input_ids[prompt_len:]
    tokenized["labels"] = labels
    return tokenized

ds = ds.map(tokenize_fn, remove_columns=["prompt", "target"])
data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

# ─── Trainer 설정 및 학습 ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=True,
    logging_steps=50,
    save_steps=200,
    save_total_limit=3,
    optim="paged_adamw_8bit",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
)

if __name__ == "__main__":
    trainer.train()
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/60 [00:00<?, ?it/s]

Map:   0%|          | 0/99999991 [00:00<?, ? examples/s]

FileNotFoundError: [Errno 2] No such file or directory: '/home/remote/.cache/huggingface/datasets/json/default-47ed5146f33c5e88/0.0.0/c8d2d9508a2a2067ab02cd118834ecef34c3700d143b31835ec4235bf10109f7/tmp2dddu_qr'